In [1]:
import os, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

FILE = "Water Quality.csv"
OUT = "task1_seasonality_outputs"
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(FILE, low_memory=False)
required = ["ActivityStartDate", "CharacteristicName", "ResultValue"]
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}; available: {list(df.columns)}")
df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
df = df.dropna(subset=required).sort_values("ActivityStartDate")
if df.empty: raise ValueError("No valid date and numeric result records.")

def safe(s): return re.sub(r"[^A-Za-z0-9_-]+", "_", str(s))[:70]
params = df["CharacteristicName"].value_counts().head(10).index
summary = []
for param in params:
    part = df[df["CharacteristicName"] == param].set_index("ActivityStartDate")["ResultValue"]
    monthly = part.resample("MS").median()
    if monthly.notna().sum() < 3: continue
    filled = monthly.interpolate(method="time", limit_direction="both")
    data = pd.DataFrame({"monthly_median": monthly,
        "rolling_3_month": filled.rolling(3, min_periods=1).mean(),
        "rolling_6_month": filled.rolling(6, min_periods=1).mean(),
        "rolling_12_month": filled.rolling(12, min_periods=1).mean()})
    name = safe(param)
    data.to_csv(f"{OUT}/{name}_monthly.csv", index_label="Date")
    plt.figure(figsize=(12,5))
    plt.plot(monthly.index, monthly, alpha=.45, label="Monthly median")
    for col in ["rolling_3_month","rolling_6_month","rolling_12_month"]:
        plt.plot(data.index, data[col], label=col.replace("_"," "))
    plt.title(f"Rolling averages: {param}"); plt.xlabel("Date"); plt.ylabel("ResultValue")
    plt.legend(); plt.tight_layout(); plt.savefig(f"{OUT}/{name}_rolling.png", dpi=220); plt.close()
    made = False
    if len(filled) >= 24 and filled.notna().all():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                dec = seasonal_decompose(filled, model="additive", period=12, extrapolate_trend="freq")
            pd.DataFrame({"observed":dec.observed,"trend":dec.trend,
                          "seasonal":dec.seasonal,"residual":dec.resid}).to_csv(f"{OUT}/{name}_decomposition.csv", index_label="Date")
            fig = dec.plot(); fig.set_size_inches(12,8); fig.suptitle(str(param)); fig.tight_layout()
            fig.savefig(f"{OUT}/{name}_decomposition.png", dpi=220, bbox_inches="tight"); plt.close(fig)
            resid = dec.resid.dropna()
            sd = resid.std()
            outliers = int((np.abs((resid-resid.mean())/sd)>3).sum()) if sd and np.isfinite(sd) else 0
            made = True
        except Exception as e:
            outliers = np.nan
            print(f"Decomposition skipped for {param}: {e}")
    else:
        outliers = np.nan
        print(f"Skipped decomposition for {param}: need at least 24 monthly points; found {len(filled)}.")
    summary.append({"Parameter":param,"Monthly_points":len(filled),
                    "Decomposition_created":made,"Residual_outliers_over_3_sd":outliers})
pd.DataFrame(summary).to_csv(f"{OUT}/seasonality_summary.csv", index=False)
print(f"Done. Outputs: {OUT}/")
print("Rolling averages smooth short-term changes. Decomposition separates trend, recurring seasonal pattern, and residuals. Gaps are interpolated for analysis; mention this limitation in your report.")

Done. Outputs: task1_seasonality_outputs/
Rolling averages smooth short-term changes. Decomposition separates trend, recurring seasonal pattern, and residuals. Gaps are interpolated for analysis; mention this limitation in your report.
